In [1]:
'''
Projeto de IA (N2) - Recomendação musical por agregação de vizinhança aprendida
Integrantes: Alex Cruz de Santana 10420357; Arthur Jones Bicalho dos Santos 10420317;
             Luis Felipe Santos do Nascimento 10420572
Síntese: limpeza, filtros, subamostra e split 80/10/10 por usuário; ajuste do ALS na validação.
Histórico:
2026-11-10 | Arthur | Criação do notebook
2026-11-13 | Luis Felipe | Célula de localização da raiz do projeto, criação de results/, estatísticas antes/depois dos filtros e ajuste do ALS
'''

'\nProjeto de IA (N2) - Recomendação musical por agregação de vizinhança aprendida\nIntegrantes: Alex Cruz de Santana 10420357; Arthur Jones Bicalho dos Santos 10420317;\n             Luis Felipe Santos do Nascimento 10420572\nSíntese: limpeza, filtros, subamostra e split 80/10/10 por usuário; ajuste do ALS na validação.\nHistórico:\n2026-11-10 | Arthur | Criação do notebook\n2026-11-13 | Luis Felipe | Célula de localização da raiz do projeto, criação de results/, estatísticas antes/depois dos filtros e ajuste do ALS\n'

In [2]:
# Localiza a raiz do repositório (pasta que contém src/) e passa a trabalhar a partir dela.
# Sem isto, abrir o notebook dentro de notebooks/ quebra os imports de src e os caminhos
# relativos data/ e results/ (FileNotFoundError).
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'data.py').exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.makedirs('results', exist_ok=True)
print('Raiz do projeto:', ROOT)

Raiz do projeto: projeto_ia


## 1. Preparação dos dados e split

In [3]:
from src.data import *
plays, prof = load_raw()
limpo = clean(plays)
df = build_interactions(plays, n_users=20000, seed=42)
df, country, users, arts = index_and_features(df, prof)
df = split_per_user(df, seed=42)

PROC.mkdir(parents=True, exist_ok=True)
df.to_parquet(PROC / 'interactions.parquet')
np.save(PROC / 'user_country.npy', country)

D = load_processed()
# verificações
tr, va, te = [set(zip(D[s]['u'], D[s]['a'])) for s in ('train', 'val', 'test')]
assert not (tr & va) and not (tr & te) and not (va & te)          # sem vazamento
assert set(D['val']['u']) == set(D['test']['u']) == set(range(D['nU']))
assert D['train'].groupby('u').size().min() >= 1

def resumo(etapa, d, ucol, acol):
    nu, na = d[ucol].nunique(), d[acol].nunique()
    return {'etapa': etapa, 'usuários': nu, 'artistas': na, 'interações': len(d),
            'esparsidade': 1 - len(d) / (nu * na),
            'artistas/usuário (mediana)': d.groupby(ucol).size().median()}
stats = pd.DataFrame([resumo('base bruta', plays.dropna(subset=['artist_name']), 'user_id', 'artist_name'),
                      resumo('após limpeza', limpo, 'user_id', 'artist'),
                      resumo('após filtros e subamostra', df, 'u', 'a')])
stats['treino/val/teste'] = ['', '', ' / '.join(str(len(D[s])) for s in ('train', 'val', 'test'))]
stats.to_csv('results/estatisticas_dataset.csv', index=False)
del plays, limpo
stats

,etapa,usuários,artistas,interações,esparsidade,artistas/usuário (mediana),treino/val/teste
0,base bruta,359349,292588,17559328,0.999833,49.0,
1,após limpeza,359349,291430,17559241,0.999832,49.0,
2,após filtros e subamostra,19653,11802,837146,0.996391,44.0,670010 / 83568 / 83568


## 2. Ajuste do ALS na validação
Grade pequena para a baseline de filtragem colaborativa não ficar mal ajustada (o que enfraqueceria a comparação).

In [4]:
from itertools import product
from src.baselines import popularity_fn, als_fn
from src.metrics import evaluate

ref = evaluate(popularity_fn(D['train_csr']), D['train_csr'], D['val_csr'], Ks=(10,))[0]['ndcg@10']
print(f'Popularidade (referência): NDCG@10 de validação = {ref:.4f}')
rows = []
for factors, reg, alpha in product((64, 128), (0.01, 0.1), (1, 20)):
    fn = als_fn(D['train_w_csr'], factors=factors, reg=reg, alpha=alpha, seed=0)
    s = evaluate(fn, D['train_csr'], D['val_csr'], Ks=(10,))[0]
    rows.append(dict(factors=factors, reg=reg, alpha=alpha, val_ndcg10=s['ndcg@10']))
als_grid = pd.DataFrame(rows).sort_values('val_ndcg10', ascending=False)
als_grid.to_csv('results/sensibilidade_als.csv', index=False)
als_grid

~/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


projeto_ia/.venv/lib/python3.12/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Popularidade (referência): NDCG@10 de validação = 0.0334


  0%|          | 0/15 [00:00<?, ?it/s]

 13%|█▎        | 2/15 [00:00<00:01,  8.40it/s]

 20%|██        | 3/15 [00:00<00:01,  7.37it/s]

... (129 outputs left out)

 93%|█████████▎| 14/15 [00:06<00:00,  2.25it/s]

100%|██████████| 15/15 [00:06<00:00,  2.14it/s]

100%|██████████| 15/15 [00:06<00:00,  2.28it/s]

,factors,reg,alpha,val_ndcg10
4,128,0.01,1,0.133509
6,128,0.10,1,0.133154
2,64,0.10,1,0.132933
0,64,0.01,1,0.132927
5,128,0.01,20,0.115125
3,64,0.10,20,0.114302
1,64,0.01,20,0.113343
7,128,0.10,20,0.113084
